# GPS-участки до начала и после конца JSON-путей

Для каждого пути из `axiom_tram_odom/json/` ищем проезды из `data_gps_union/` в корне репозитория с обеих сторон: **от кольца к началу JSON** и **от конца JSON к кольцу**. Трек должен пройти стык с ошибкой до 15 м, совпасть с JSON-путём на расстоянии около 200 м внутри него и продолжиться до района кольца минимум на 120 м. Начальные или конечные GPS-точки должны попасть в общий кластер радиусом 80 м. Точки с переходом быстрее 30 м/с отбрасываются. Кластер — оценка района кольца, а не его готовая геометрия.

На картах линии показаны целиком; начальный масштаб ограничен только районом соответствующего стыка и кольца.

Для отбора требуется, чтобы не менее 75% длины GPS-линии проходило в пределах 8 м от известного JSON-пути, а совпавшая длина составляла не менее 75% длины JSON. При поиске кандидатов GPS-геометрия не сдвигается; перенос выполняется только при сборке продолженных JSON.

## Полный алгоритм продолжения JSON

1. Для начала и конца каждого исходного JSON ищутся GPS-проезды, которые подходят к стыку и совпадают с известной линией внутри маршрута.
2. В списках после карт указываются ID проездов, которые нужно использовать для каждого края.
3. Для каждого GPS-трека координаты переводятся из GPS в систему JSON. Его точка стыка переносится на крайнюю точку исходного пути одним постоянным сдвигом; высота GPS дополнительно привязывается к высоте в точке стыка.
4. Самый длинный выбранный проезд задаёт длину и основную форму продолжения. Остальные проезды сравниваются с ним на одинаковом расстоянии от стыка; их отклонения усредняются. Вклад более короткого проезда плавно уменьшается на последних 25 м.
5. Усреднённые отклонения сглаживаются по расстоянию. Исходные точки JSON сохраняются; для новых точек вычисляются высота, касательная и кривизна.
6. Продолженные файлы записываются в `json/continued/`, после чего карта показывает исходные JSON, продолженные JSON и выбранные GPS-треки.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
notebook_candidates = (
    cwd,
    cwd / "notebooks",
    cwd / "path_research" / "notebooks",
    cwd / "axiom_tram_odom" / "path_research" / "notebooks",
)
NOTEBOOKS_DIR = next(
    candidate for candidate in notebook_candidates
    if (candidate / "tools").is_dir()
)
sys.path.insert(0, str(NOTEBOOKS_DIR))

from tools.continuations import find_endpoint_extensions
from tools.extend_path_graphs import save_continued_graph
from tools.gps_tracks import load_gps_track
from tools.path_graphs import load_path_graph_tracks
from tools.track_map import show_tracks_map

PATH_RESEARCH_DIR = NOTEBOOKS_DIR.parent
PACKAGE_DIR = PATH_RESEARCH_DIR.parent
REPOSITORY_DIR = PACKAGE_DIR.parent
GRAPH_DIR = PACKAGE_DIR / "json"
UNION_DIR = REPOSITORY_DIR / "data_gps_union"
OUTPUT_DIR = GRAPH_DIR / "continued"
ROUTE_FILES = ("shykinskaya-talinskaya.json", "talinskaya-shykinskaya.json")


In [ ]:
matches = find_endpoint_extensions(GRAPH_DIR, UNION_DIR)

for graph_name, sides in matches.items():
    print(f"\n{graph_name}")
    for side, label in (("start", "До начала"), ("end", "После конца")):
        result = sides[side]
        ring = result["ring_xy"]
        print(f"  {label}: {len(result['candidates'])} кандидатов")
        if ring is not None:
            print(f"  Конечный кластер: x={ring[0]:.1f}, y={ring[1]:.1f}")
        print("  ID                    Совп., %  Стык, м  Внутри, м  Участок, м  До кластера, м")
        for candidate in result["candidates"]:
            print(
                f"  {candidate['id']:<20} {candidate['overlap_ratio'] * 100:8.1f} {candidate['junction_gap_m']:7.1f}"
                f" {candidate['inside_gap_m']:10.1f} {candidate['extension_m']:11.0f}"
                f" {candidate['ring_gap_m']:15.1f}"
            )


In [ ]:
graph_tracks = {track["id"]: track for track in load_path_graph_tracks(GRAPH_DIR)}

def show_extension_map(graph_name, side):
    candidates = matches[graph_name][side]["candidates"][:5]
    if not candidates:
        print(f"Для {graph_name}, {side}: кандидатов нет")
        return None

    graph = graph_tracks[graph_name]
    graph_near_junction = graph["points"][:250] if side == "start" else graph["points"][-250:]
    focus_points = graph_near_junction + [
        point for candidate in candidates for point in candidate["extension_track"]["points"]
    ]
    south = min(point[0] for point in focus_points)
    west = min(point[1] for point in focus_points)
    north = max(point[0] for point in focus_points)
    east = max(point[1] for point in focus_points)
    lat_margin = max((north - south) * 0.08, 0.0001)
    lon_margin = max((east - west) * 0.08, 0.0001)
    focus_bounds = [
        [south - lat_margin, west - lon_margin],
        [north + lat_margin, east + lon_margin],
    ]
    return show_tracks_map(
        [graph] + [candidate["full_track"] for candidate in candidates],
        height=450,
        focus_bounds=focus_bounds,
    )


## Щукинская → Таллинская: до начала JSON

Показаны полные JSON-путь и пять полных GPS-треков с наибольшей долей совпадения; карта сфокусирована на области начала пути и кольца.


In [ ]:
shch_start_map = show_extension_map("shykinskaya-talinskaya.json", "start")


In [ ]:
# ID GPS-треков для продолжения: Щукинская → Таллинская, до начала
SHCH_START_IDS = [
    "30618_b83d854d",
    "30618_dd8e6395",
    "30618_437e855c",
    "30618_0e41eac3",
]  # Например: ["30618_..."]


## Щукинская → Таллинская: после конца JSON

Показаны полные JSON-путь и пять полных GPS-треков с наибольшей долей совпадения; карта сфокусирована на области конца пути и кольца.


In [ ]:
shch_end_map = show_extension_map("shykinskaya-talinskaya.json", "end")


In [ ]:
# ID GPS-треков для продолжения: Щукинская → Таллинская, после конца
SHCH_END_IDS = [
    "30618_095a115b",
    "30618_2dbce472",
    "30618_3b36e5cd",
]  # Например: ["30618_..."]


## Таллинская → Щукинская: до начала JSON

Показаны полные JSON-путь и пять полных GPS-треков с наибольшей долей совпадения; карта сфокусирована на области начала пути и кольца.


In [ ]:
tall_start_map = show_extension_map("talinskaya-shykinskaya.json", "start")


In [ ]:
# ID GPS-треков для продолжения: Таллинская → Щукинская, до начала
TALL_START_IDS = [
    "30618_e2dcf65f",
    "30618_d4ba7005",
    "30639_927002c2",
]  # Например: ["30618_..."]


## Таллинская → Щукинская: после конца JSON

Показаны полные JSON-путь и пять полных GPS-треков с наибольшей долей совпадения; карта сфокусирована на области конца пути и кольца.


In [ ]:
tall_end_map = show_extension_map("talinskaya-shykinskaya.json", "end")


In [ ]:
# ID GPS-треков для продолжения: Таллинская → Щукинская, после конца
TALL_END_IDS = [
    "30618_d4ba7005",
    "30618_33bec73f",
    "30618_2050d396",
]  # Например: ["30618_..."]


## Сохранение продолженных JSON

Заполните списки ID после карт и выполните эту ячейку. Для каждого края самый длинный выбранный участок задаёт длину и форму; остальные усредняются с ним по расстоянию от стыка. На последних 25 м каждого короткого трека его вклад плавно снижается, чтобы конец трека не создавал излом. Если список для одного из краёв пуст, соответствующий JSON не сохраняется.

После усреднения новые участки сглаживаются в пределах 20 м. Точки стыка с исходным JSON и крайние точки у колец остаются на месте.

Для любого числа ID весь GPS-участок целиком переносится к стыку JSON одним и тем же сдвигом. Самый длинный трек задаёт форму, а сглаживание применяется лишь к отклонениям остальных треков от неё.


In [ ]:
SMOOTHING_RADIUS_M = 20  # метры; 0 отключает сглаживание

selections = {
    "shykinskaya-talinskaya.json": (SHCH_START_IDS, SHCH_END_IDS),
    "talinskaya-shykinskaya.json": (TALL_START_IDS, TALL_END_IDS),
}

for graph_name, (start_ids, end_ids) in selections.items():
    if not start_ids or not end_ids:
        print(f"{graph_name}: заполните ID для начала и конца; сохранение пропущено")
        continue
    output_file = OUTPUT_DIR / graph_name
    info = save_continued_graph(
        GRAPH_DIR / graph_name, UNION_DIR, output_file,
        start_ids, end_ids, matches[graph_name],
        smoothing_radius_m=SMOOTHING_RADIUS_M,
    )
    print(
        f"{output_file}: +{info['added_start']} точек в начале "
        f"(основа {info['start_reference']}), +{info['added_end']} в конце "
        f"(основа {info['end_reference']})"
    )


## Итоговые пути на карте

На одной карте показаны продолженные JSON, исходные JSON и полные GPS-треки из выбранных списков ID. Повторяющийся GPS ID добавляется один раз. Слои можно включать и выключать справа.


In [ ]:
selected_gps_ids = list(dict.fromkeys(
    bag_id
    for start_ids, end_ids in selections.values()
    for bag_id in (*start_ids, *end_ids)
))

continued_tracks = [
    {"id": f"Продолженный JSON · {track['id']}", "points": track["points"]}
    for track in load_path_graph_tracks(OUTPUT_DIR)
    if track["id"] in selections
]
original_tracks = [
    {"id": f"Исходный JSON · {track['id']}", "points": track["points"]}
    for track in load_path_graph_tracks(GRAPH_DIR)
    if track["id"] in selections
]
selected_gps_tracks = [
    {"id": f"GPS · {bag_id}", "points": load_gps_track(
        UNION_DIR, bag_id, max_points=1_200, max_speed_m_s=30,
    )["points"]}
    for bag_id in selected_gps_ids
]

continued_map = show_tracks_map(
    continued_tracks + original_tracks + selected_gps_tracks,
    height=550,
)
